# Проект спринта 15: Модель поведенческого скоринга для банка "Ва-банк"

Выполнил: Артем Буров (DS12)  
Дата: 08.08.2026

# Описание проекта

**Банк «Ва‑банк»** — крупный розничный игрок с многомиллиардным кредитным портфелем: ежедневно в него поступают тысячи заявок, а совокупные активы исчисляются миллиардами рублей. При внешней стабильности бизнеса в сегменте потребительского кредитования нарастает скрытая угроза — волна просрочек, которая действует по принципу айсберга: видимая часть — это отдельные случаи задержек платежей, а основная опасность кроется в системных последствиях, способных серьёзно подорвать финансовую устойчивость.

**Ключевая проблема** — рост просроченной задолженности длительностью свыше 90 дней. По требованиям ЦБ банк обязан ежемесячно резервировать 100 % суммы таких проблемных кредитов, что приводит к масштабной заморозке капитала и сокращению свободных средств в обороте. Из‑за этого снижается ликвидность: резервы формируются за счёт прибыли, и банк вынужден ограничивать выдачу новых кредитов, теряя гибкость в управлении активами. Регуляторные риски дополнительно усиливают давление: несоблюдение нормативов грозит штрафами, предписаниями и даже отзывом лицензии. Поэтому своевременное выявление потенциально проблемных заёмщиков — это не просто инструмент оптимизации, а вопрос финансовой безопасности банка.

Действующие скоринговые модели оценивают риск только на этапе выдачи кредита и не позволяют отслеживать динамику платёжного поведения уже действующих клиентов: они не учитывают постепенное ухудшение финансового состояния, которое может проявиться спустя месяцы после получения займа. Именно поэтому банк нуждается в проактивном инструменте, который позволит заранее выявлять клиентов с высокой вероятностью серьёзной просрочки и управлять резервами и ликвидностью, избегая резких колебаний и непредвиденных потерь.

При этом данные имеют панельную структуру (клиент × месяц), а прогноз строится на горизонт в 12 месяцев — это требует строгого соблюдения временного порядка при разбиении выборки, чтобы исключить утечку информации из будущего. Ожидается значительный дисбаланс целевой переменной (просрочка ≥ 90 дней — относительно редкое событие), что диктует особый подход к выбору метрик качества и способам работы с классами. Кроме того, для банка критически важна интерпретируемость модели: выводы должны быть понятны не только аналитикам, но и регулятору — поэтому важно обеспечить возможность объяснения прогнозов на уровне признаков.

# Цель проекта

Предсказать вероятность того, что клиент банка совершит просрочку платежа по кредиту длительностью от 90 дней. Объект моделирования — клиент в конкретный месяц; целевая переменная — флаг, будет ли у клиента просрочка от 90 дней в течение ближайших 12 месяцев.

# Задача

Построить модель поведенческого скоринга, которая по данным о клиенте в выбранный месяц прогнозирует, возникнет ли у него просрочка на 90 дней или более. Каждая строка данных соответствует клиенту в конкретном месяце. Целевая переменная принимает значение 1, если в ближайшие 12 месяцев у клиента появляется просрочка длиной от 90 дней, и 0 — если такой просрочки не возникает.

# Метрики

## Recall («полнота»)

Метрика показывает, какую долю объектов класса 1 модель смогла корректно предсказать. В задаче поведенческого скоринга это доля клиентов с высоким риском дефолта, которых удалось определить.

### Формула расчёта

$$
\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}
$$

Где:

- **TP** — количество клиентов, которых модель посчитала рискованными (присвоила им класс 1) и у которых действительно случился дефолт.
- **FN** — количество клиентов, которых модель посчитала надёжными (присвоила им класс 0), но у них случился дефолт.

### Интерпретация

- **Высокое значение Recall** означает, что модель корректно предсказывает дефолт для подавляющего большинства клиентов в зоне риска — банк «видит» почти всех проблемных заёмщиков.
- **Низкое значение метрики** сигнализирует о том, что многие потенциальные неплательщики остались незамеченными: это критично для банка, так как ведёт к недорезервированию и регуляторным рискам.
___

## Precision («точность»)

Метрика показывает, какая доля объектов, которым был присвоен класс 1, действительно к нему относится. В бизнес‑контексте проекта precision покажет, какую долю клиентов, помеченных как рискованные, составляют действительно проблемные заёмщики.

### Формула расчёта

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}
$$

Где:

- **TP** — количество клиентов, которых модель посчитала рискованными (присвоила им класс 1) и у которых действительно случился дефолт.
- **FP** — количество клиентов, которых модель посчитала рискованными (присвоила им класс 1), но у которых дефолт не случился.

### Интерпретация

- **Высокое значение Precision** означает, что среди тех клиентов, которых модель пометила как рискованных, почти все действительно ушли в дефолт.
- **Низкое значение метрики** означает, что среди клиентов, которых модель определяет в зону риска, много тех, кто на самом деле выплачивает платежи вовремя.

___

## Approval rate («уровень одобрения»)

Метрика показывает долю клиентов, которых модель считает надёжными — не видит у них риска дефолта. Она позволяет оценить, скольким клиентам будет присвоен класс 0 при оценке риска просрочки.

### Формула расчёта

$$
\text{Approval rate} = \frac{\text{FN} + \text{TN}}{\text{N}}
$$

Где:

- **FN** — количество клиентов, которых модель посчитала надёжными (присвоила им класс 0), но у них случился дефолт.
- **TN** — количество клиентов, которых модель посчитала надёжными (присвоила им класс 0) и у которых дефолта не случилось.
- **N** — размер всей выборки заявок или текущих заёмщиков.

### Интерпретация

**Высокое значение approval rate** означает, что модель даёт одобрение большому количеству клиентов — то есть предполагает, что просрочек не будет. Бизнес будет рад такому сценарию, если модель действительно корректно определяет надёжных клиентов. Однако это создаёт риски: если модель слабая, при таком уровне одобрений много рискованных клиентов останутся незамеченными. Банку будет казаться, что всё в порядке, но накопленный риск может обнаружиться позже.

**Низкое значение approval rate** означает, что модель одобряет только самых надёжных заёмщиков. Круг клиентов получается безопасным, но такой подход сильно урезает объём бизнеса: банк сознательно жертвует доходом, так как часть платёжеспособных клиентов получает отказ. Из‑за этого часть денег банка не попадёт в оборот, и потенциал прибыли не реализуется.

___

## Default rate («доля просрочек платежа»)

Метрика показывает, какая часть клиентов среди тех, кого модель посчитала надёжными, в итоге всё‑таки не заплатила по своим обязательствам. Другими словами, это доля клиентов среди одобренных моделью, которые всё‑таки ушли в дефолт.

### Формула расчёта

$$
\text{Default rate} = \frac{\text{FN}}{\text{FN} + \text{TN}}
$$

Где:

- **FN** — количество клиентов, которых модель посчитала надёжными (класс 0), но у них случился дефолт.
- **TN** — количество клиентов, которых модель посчитала надёжными (класс 0) и у которых дефолта не случилось.

### Интерпретация

- **Высокое значение Default rate** означает, что внутри группы заёмщиков, которая считается надёжной и получает одобрение, заметная доля клиентов всё‑таки попадает в дефолт. Формально модель выглядит «мягкой»: она даёт много одобрений, но среди них сидит слишком много будущих проблемных клиентов.
- **Низкое значение Default rate** означает, что среди тех, кого модель обозначила как надёжных, почти все действительно платят. Такая модель мало ошибается, и её прогнозы относительно безопасны для банка.

___

## Missed defaults rate («доля пропущенных дефолтов»)

Метрика показывает, какую долю от всех проблемных клиентов модель не заметила — то есть посчитала этих клиентов надёжными и пропустила дальше, хотя они в итоге не заплатили.

### Формула расчёта

$$
\text{Missed defaults rate} = \frac{\text{FN}}{\text{FN} + \text{TP}}
$$

Также Missed defaults rate можно выразить как $1 - \text{Recall}$.

Где:

- **FN** — количество клиентов, которых модель посчитала надёжными (класс 0), но у них случился дефолт.
- **TP** — количество клиентов, которых модель посчитала рискованными (класс 1) и у которых действительно случился дефолт.

### Интерпретация

- **Высокое значение Missed defaults rate** означает, что модель отмечает заметную долю будущих неплательщиков как надёжных клиентов. Для банка это создаёт опасную ситуацию: формально всё будет спокойно, но внутри множества одобренных клиентов будет расти количество рискованных заёмщиков — это проявится позже в виде просрочек и списаний. Чем выше эта метрика, тем больше дефолтов будет пропускать банк.
- **Низкое значение Missed defaults rate** означает, что из всех клиентов, которые в итоге ушли в дефолт, лишь небольшая часть была отмечена моделью как надёжные. Большинство проблемных случаев модель отфильтровывает.

___




## ???????????????? Перевод бизнес-задачи на язык машинного обучения

* Напишите, что конкретно будете делать, пользуясь терминами машинного обучения:
  * какие модели будете использовать;
  * какие методы работы с данными применять;
  * как планируете решать поставленную задачу.


## Загрузка необходимых библиотек

In [1]:
# Загружаем базовые библиотеки
import numpy as np
import pandas as pd


# Системные библиотеки, настройки отображения датафреймов и предупредительных сообщений
# import joblib
import os
import re
# import warnings

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

# warnings.filterwarnings('ignore', category=FutureWarning)
# warnings.filterwarnings('ignore', category=UserWarning)
# warnings.filterwarnings('ignore', category=DeprecationWarning)

path = os.getcwd()
RANDOM_SEED = 555

## Подготовка классов и функций

In [2]:
# Подготавливаем функции визуализации данных и расчета метрик

# =========================
# Функция оптимизации датафрейма

def optimize_dataframe(df: pd.DataFrame) -> pd.DataFrame:

    # 1. Приводим названия колонок к нижнему регистру
    df.columns = df.columns.astype(str).str.lower()

    # 2. Удаляем спецсимволы, удаляем пробелы по краям: оставляем только буквы, цифры
    # [^a-z0-9_] — всё, что не является латинской буквой, цифрой или подчёркиванием
    df.columns = df.columns.str.replace(r'[^a-z0-9_]', ' ', regex=True).str.strip()
    # 3. Удаляем пробелы, заменяем на _
    df.columns = df.columns.str.replace(r'\s+', '_', regex=True)   
  

    # На случай, если после очистки остались пустые названия — заменяем их на col_N
    empty_cols = df.columns == ''
    if empty_cols.any():
        empty_indices = df.columns[empty_cols].index
        for i, idx in enumerate(empty_indices):
            df.columns.values[i] = f'col_{i}'

    # 3. Оптимизация типов для float
    float_cols = df.select_dtypes(include=['float64', 'float32']).columns.tolist()
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], downcast='float')

    # 4. Оптимизация типов для int
    int_cols = df.select_dtypes(include=['int64', 'int32']).columns.tolist()
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    return df

# =========================
# Анализ пропущенных значений

def show_missing_stats(df):
    """
    Функция для отображения статистики пропущенных значений в DataFrame.
    """
    missing_stats = pd.DataFrame({
        'Кол-во пропусков': df.isnull().sum(),
        'Доля пропусков': df.isnull().mean()
    }).sort_values(by='Кол-во пропусков', ascending=False)
    missing_stats = missing_stats[missing_stats['Кол-во пропусков'] > 0]
    
    if missing_stats.empty:
        return "Пропусков в данных нет"
    
    # Форматируем при выводе через Styler
    return (missing_stats.style.format({'Доля пропусков': '{:.4f}'}).background_gradient(cmap='coolwarm'))

## Загрузка данных

* Загрузите предоставленные датасеты. Обратите внимание на то, как загрузятся данные, в каком формате представлены таблицы. Пути к данным:
  * `'/datasets/ds_15_loan_payment_credit.csv'`;
  * `'/datasets/ds_15_transactions.csv'`;
  * `'/datasets/ds_15_client_description.csv'`;
  * `'/datasets/ds_15_credit_description.csv'`;
  * `'/datasets/ds_15_mortgage_presence.csv'`;
  * `'/datasets/ds_15_credit_rating.csv'`;
  * `'/datasets/ds_15_macro_data.csv'`;
  * `'/datasets/ds_15_cohort_grid.csv'`.
* Посмотрите на содержимое таблиц. Сделайте первые выводы о них.

## Исследовательский анализ данных

* Проведите первичный анализ данных:
    * Проверьте разные характеристики данных.
    * Исследуйте с помощью графиков количественные и категориальные данные.
    * Рекомендуем создать для этого функции, но это необязательное требование.

* Сделайте выводы о выбросах, пропусках, дубликатах и иных аномалиях в данных из каждой таблицы.

* Предобработка данных или их трансформация в этом проекте необязательны — кроме действий, необходимых для объединения таблиц.

## Объединение таблиц

Соберите все источники данных о клиентах в единую таблицу наблюдений.

### Формирование целевой переменной

1. Значение бинарной целевой переменной нужно определить для каждой строки со столбцами `ID` и `score_date` в таблице `cohort_grid`.

2. Таргет равен 1 при соблюдении двух условий:
    * Если значение в поле `просрочка_дней` больше или равно 90.
    * Если для клиента существует строка в таблице `loan_payment_credit`, где значение в поле `дата_начала_периода` попадает в интервал `[score_date, score_date + 365 дней)`.

>Важно: у клиента может быть несколько эпизодов с просрочками от 90 дней. Вам нужно взять первый по времени возникновения эпизод в таблице с просрочками.

3. После расчёта целевой переменной удалите строки, где дефолт уже произошёл к моменту скоринга, то есть `дата_начала_периода < score_date`. Это необходимо, так как для корректной работы с временной структурой важно учитывать дефолты, произошедшие в прошлом относительно даты скоринга.

### Создание итоговой таблицы

1. В качестве признаков можно использовать только информацию о прошлом, то есть она должна быть доступна к дате скоринга. Иными словами, в каждой строке нужно присоединить данные о поведении клиента за предыдущие периоды относительно даты скоринга, иначе произойдёт утечка данных из будущего.

>К примеру, `score_date = 2024-01-15`. Тогда:
>* транзакции за декабрь 2023 г. — можем использовать;
>* транзакции за ноябрь 2023 г. — можем использовать;
>* транзакции 16 января 2024 г. — **не** можем использовать.

2. Присоедините остальные данные по клиенту, помимо указанных выше данных о макроэкономике и транзакциях клиента.

> Рекомендации:
>* Не забывайте проверять правильность каждого этапа сбора данных в единую таблицу. Это можно отслеживать на одном из клиентов.
>* При формировании таблицы следите за тем, чтобы в ней была корректно проведена работа со временем:
  >   * Отследите, не упущены ли какие-то данные из прошлого;
  >   * Проконтролируйте, верно ли рассчитана целевая переменная, которая зависит от дефолта в будущем.
>* Помните, что даты в исходных таблицах указаны на первое число месяца. Учитывайте период, который они описывают.

Сделайте выводы о получившейся таблице.

## Создание новых признаков

* Добавьте в таблицу новые признаки, которые помогли бы описать поведение клиента. Создайте не менее двух новых признаков.
* Сделайте выводы о новых признаках.

## Анализ итоговой таблицы

* Проведите краткий анализ получившейся итоговой таблицы.
* Сделайте вывод о данных для моделирования.
* Проверьте целевую переменную на предмет дисбаланса классов. Сделайте выводы.

## Моделирование

### Базовые модели

1. Подготовьте обучающую, калибровочную и тестовую выборки. Разбейте обучающую на три фолда для последующего использования кросс-валидации. Для оценки качества и калибровки используйте размер выборки, равный 12 месяцам.


2. При необходимости проведите категоризацию данных, применив нужный Encoder и использовав пайплайн.

3. Обучите базовые модели с кросс-валидацией по трём фолдам:
    * Две базовые модели — логистическую регрессию и случайный лес — без балансировки классов в целевой переменной.
    * Логистическую регрессию и случайный лес с балансировкой классов. Выберите метод балансировки самостоятельно. Обязательно примените хотя бы один метод. Можно попробовать несколько и выбрать лучший.
    * Сделайте выводы о работе всех четырёх моделей.

4. Случайный лес с настройками по умолчанию легко переобучается, потому что запоминает обучающую выборку, из-за чего модель может терять в качестве на новых данных. Логистическая регрессия же сразу готова к работе за счёт встроенной L2-регуляризации, которая автоматически контролирует сложность модели.

   Чтобы исправить проблемы модели Random Forest, вам нужно подобрать для неё гиперпараметры с помощью  Optuna. Количество гиперпараметров должно быть не менее трёх. Для оптимизации используйте метрику missed defaults rate.

5. Сравните все полученные модели.

6. Для оценки моделей используйте метрики:
   * accuracy или ROC-AUC,
   * approval rate,
   * default rate,
   * missed defaults rate.

7. Сделайте вывод о работе, проделанной в этом разделе.

## Калибровка модели и пересчёт результатов

* Проведите калибровку лучшей версии модели. Используйте отдельную калибровочную выборку.
* Используйте метод, подходящий для случайного леса.
* Постройте график калибровки.
* Сделайте вывод, оцените результаты с помощью коэффициента Бриера.

## Поиск порога решения

* Используя откалиброванную модель и калибровочную выборку, найдите порог, при котором будут достигнуты заданные в постановке задачи значения метрик:
    * approval rate — не менее 65%;
    * default rate — не более 2%;
    * missed defaults rate — не более 4%.
    
* Сделайте вывод о достигнутых в этом разделе результатах.

## Анализ матрицы ошибок

* Оцените стабильность модели на тестовых данных. Для этого постройте:
    * матрицу ошибок на калибровочных данных;
    * матрицу классификации на тестовых данных.
* Сделайте вывод о моделях, рассчитав классические метрики машинного обучения и указанные в ТЗ бизнес-метрики.
* Сделайте вывод о стабильности модели.

## Фиксирование итоговой модели

- Опишите лучшую модель и найденный порог классификации.


## Анализ важности признаков

* Проведите анализ важности признаков найденной модели на полных тренировочных данных.
* Используйте `feature_importances_` для найденной модели.
* Сделайте вывод о силе влияния признаков на дефолт.

## Выводы по проекту

Сделайте выводы по проекту. Можете использовать такой план:

1. Цель и задачи исследования.

2. Подготовка данных и выборок.

3. Поиск и настройка модели.

4. Калибровка вероятностей.

5. Оптимизация бизнес-порога.

6. Анализ важности признаков.

7. Финальный пайплайн.

8. Основные выводы и рекомендации для бизнеса.

